# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Role of Model Evaluation
This is the tenth stage of the pipeline. In previous stages, we trained baseline models (Stage 07) and theoretically improved them using Hyperparameter Tuning (Stage 08) based *only* on the training dataset. Now, we must evaluate them against unseen data.

## Strict Test Set Principle
To guarantee absolute research integrity, this notebook evaluates the models using the held-out **Test Dataset**. This dataset has never been touched by the TF-IDF vectorizer's vocabulary builder or the models' training loops. The performance metrics generated here represent how the models will behave in the real world.


# 2. MODEL EVALUATION OBJECTIVES

Our goals for this stage are to:
- Load the fully isolated Test TF-IDF matrices and target labels.
- Load all 6 trained models (3 Baseline, 3 Tuned).
- Calculate robust evaluation metrics (Accuracy, Precision, Recall, F1-Score, ROC-AUC) prioritizing Macro-averaging due to class imbalance.
- Generate visualization artifacts (Confusion Matrices).
- Compare all models side-by-side.
- Automatically select the final, absolute best model for Indonesian Cyberbullying text classification based on the highest Macro F1-Score.


In [1]:
# 3. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import pathlib
import json
import joblib
from scipy import sparse
import warnings

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
TFIDF_DIR = PROJECT_ROOT / "data" / "processed" / "tfidf"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

# Test Data Paths
X_TEST_PATH = TFIDF_DIR / "X_test_tfidf.npz"
Y_TEST_PATH = TFIDF_DIR / "y_test.csv"
XGB_MAPPING_PATH = MODELS_DIR / "xgboost_label_mapping.json"

# Averaging Strategy
AVG_STRATEGY = 'macro'  # Crucial for imbalanced multi-class cyberbullying

# List of models to evaluate
MODEL_NAMES = [
    "logistic_regression_baseline",
    "linear_svm_baseline",
    "xgboost_baseline",
    "logistic_regression_type_tuned",
    "linear_svm_type_tuned",
    "xgboost_type_tuned"
]

print(f"Reports Directory: {REPORTS_DIR}")
print(f"Averaging Strategy: {AVG_STRATEGY}")


Reports Directory: /home/zapp/Kampus/PM-NEW/reports
Averaging Strategy: macro


In [3]:
# 5. LOAD TF-IDF TEST DATA

if not X_TEST_PATH.exists() or not Y_TEST_PATH.exists():
    raise FileNotFoundError("FAIL: Test artifacts not found. Please run previous stages first.")

print("Loading Unseen Test Data...")
X_test = sparse.load_npz(X_TEST_PATH)
y_test_df = pd.read_csv(Y_TEST_PATH)
y_test = y_test_df.iloc[:, 0]  # Pandas Series

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


Loading Unseen Test Data...
X_test shape: (6443, 35222)
y_test shape: (6443,)


In [4]:
# 6. LOAD MODELS

print("Loading trained models...")
models = {}

for name in MODEL_NAMES:
    model_path = MODELS_DIR / f"{name}.pkl"
    if not model_path.exists():
        raise FileNotFoundError(f"Model file not found: {model_path}. Please ensure stage 07 and 08 finished successfully.")
    
    models[name] = joblib.load(model_path)
    print(f"[OK] Loaded: {name}")
    
# Load XGBoost label mapping to decode its integer predictions back to string classes
with open(XGB_MAPPING_PATH, 'r') as f:
    xgb_mapping = json.load(f)
xgb_reverse_mapping = {v: k for k, v in xgb_mapping.items()}

print("\nAll models loaded successfully.")


Loading trained models...
[OK] Loaded: logistic_regression_baseline
[OK] Loaded: linear_svm_baseline
[OK] Loaded: xgboost_baseline
[OK] Loaded: logistic_regression_type_tuned
[OK] Loaded: linear_svm_type_tuned
[OK] Loaded: xgboost_type_tuned

All models loaded successfully.


In [5]:
# 7. VALIDATE FEATURE AND LABEL ALIGNMENT

if X_test.shape[0] != y_test.shape[0]:
    raise ValueError(f"CRITICAL ERROR: Misalignment! X_test has {X_test.shape[0]} rows, y_test has {y_test.shape[0]}.")

print("Test Features and Labels alignment validated.")


Test Features and Labels alignment validated.


In [6]:
# 8. INSPECT TARGET LABELS

class_counts = y_test.value_counts()
class_percentages = y_test.value_counts(normalize=True) * 100

print(f"Number of classes in Test Set: {len(class_counts)}\n")
print("Target Labels Distribution (Test Data):")
for class_name, count in class_counts.items():
    print(f"- {class_name:<15}: {count} ({class_percentages[class_name]:.2f}%)")


Number of classes in Test Set: 4

Target Labels Distribution (Test Data):
- normal         : 4337 (67.31%)
- hate_speech    : 1072 (16.64%)
- harassment     : 772 (11.98%)
- insult         : 262 (4.07%)


In [7]:
# 9. DEFINE EVALUATION METRICS

def calculate_metrics(y_true, y_pred, y_prob=None, model_name=""):
    metrics = {}
    
    # Standard Metrics
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Precision'] = precision_score(y_true, y_pred, average=AVG_STRATEGY, zero_division=0)
    metrics['Recall'] = recall_score(y_true, y_pred, average=AVG_STRATEGY, zero_division=0)
    metrics['F1-Score'] = f1_score(y_true, y_pred, average=AVG_STRATEGY, zero_division=0)
    
    # ROC-AUC (Requires probabilities or decision scores)
    if y_prob is not None:
        try:
            metrics['ROC-AUC'] = roc_auc_score(y_true, y_prob, average=AVG_STRATEGY, multi_class='ovr')
        except ValueError as e:
            metrics['ROC-AUC'] = np.nan
            print(f"Warning: Could not calculate ROC-AUC for {model_name} - {str(e)}")
    else:
        metrics['ROC-AUC'] = np.nan
        
    return metrics

print("Evaluation metrics formulas initialized.")


Evaluation metrics formulas initialized.


In [8]:
# 10. GENERATE PREDICTIONS & 11. CALCULATE CLASSIFICATION METRICS & 12. CALCULATE ROC-AUC

# We will loop through all models, generate predictions, and calculate metrics simultaneously.
evaluation_results = []
predictions_dict = {}

print("Evaluating Models on Test Set...")

for name, model in models.items():
    
    # 1. Predictions
    y_pred_raw = model.predict(X_test)
    
    # XGBoost requires mapping back from int to string
    if 'xgboost' in name:
        y_pred = pd.Series(y_pred_raw).map(xgb_reverse_mapping).values
    else:
        y_pred = y_pred_raw
        
    predictions_dict[name] = y_pred
    
    # 2. Probability Scores for ROC-AUC
    y_prob = None
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
    elif hasattr(model, "decision_function"):
        # LinearSVC doesn't have predict_proba natively, we use decision_function for ranking
        y_prob = model.decision_function(X_test)
        
    # XGBoost probability columns need to be aligned with actual string classes conceptually,
    # but roc_auc_score(multi_class='ovr') handles the array matrix directly if y_true is encoded.
    # To keep things simple and avoid complex re-mapping for AUC calculation, we encode y_test temporarily
    y_true_for_auc = y_test
    if 'xgboost' in name:
        y_true_for_auc = y_test.map(xgb_mapping).values

    # 3. Calculate
    res = calculate_metrics(y_true_for_auc if 'xgboost' in name else y_test, y_pred_raw if 'xgboost' in name else y_pred, y_prob, name)
    res['Model'] = name
    evaluation_results.append(res)
    
    print(f"[DONE] {name}")

print("\nPredictions and metric calculations completed.")


Evaluating Models on Test Set...
[DONE] logistic_regression_baseline
[DONE] linear_svm_baseline
[DONE] xgboost_baseline
[DONE] logistic_regression_type_tuned
[DONE] linear_svm_type_tuned
[DONE] xgboost_type_tuned

Predictions and metric calculations completed.


In [9]:
# 13. GENERATE CLASSIFICATION REPORTS

reports_dict = {}

for name, y_pred in predictions_dict.items():
    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    reports_dict[name] = rep

print("Classification reports generated (available in reports_dict for detailed inspection).")


Classification reports generated (available in reports_dict for detailed inspection).


In [10]:
# 14. GENERATE CONFUSION MATRICES

def plot_confusion_matrix(y_true, y_pred, model_name):
    # Ensure consistent label ordering
    labels = sorted(y_true.unique())
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    
    plt.title(f'Confusion Matrix - {model_name}', pad=20, fontsize=14)
    plt.ylabel('Actual Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save to reports/
    file_name = f"confusion_matrix_{model_name}.png"
    save_path = REPORTS_DIR / file_name
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close() # Close to prevent overwhelming the notebook display
    
    return save_path

print("Generating and saving Confusion Matrices...")
cm_paths = []
for name, y_pred in predictions_dict.items():
    path = plot_confusion_matrix(y_test, y_pred, name)
    cm_paths.append(path)
    print(f"Saved: {path.name}")


Generating and saving Confusion Matrices...
Saved: confusion_matrix_logistic_regression_baseline.png
Saved: confusion_matrix_linear_svm_baseline.png
Saved: confusion_matrix_xgboost_baseline.png
Saved: confusion_matrix_logistic_regression_type_tuned.png
Saved: confusion_matrix_linear_svm_type_tuned.png
Saved: confusion_matrix_xgboost_type_tuned.png


In [11]:
# 15. COMPARE MODEL PERFORMANCE

# Create a Master Comparison DataFrame
df_results = pd.DataFrame(evaluation_results)

# Reorder columns
df_results = df_results[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']]

# Sort by F1-Score (Macro) descending
df_results = df_results.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

print("--- MASTER MODEL COMPARISON ON UNSEEN TEST DATA ---")
display(df_results.round(4))


--- MASTER MODEL COMPARISON ON UNSEEN TEST DATA ---


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,linear_svm_baseline,0.7956,0.6717,0.6670,0.6688,NaN
1,linear_svm_type_tuned,0.7956,0.6717,0.6670,0.6688,NaN
2,logistic_regression_type_tuned,0.7743,0.6476,0.6838,0.6639,0.9085
3,logistic_regression_baseline,0.7467,0.6267,0.7227,0.6604,0.9158
4,xgboost_type_tuned,0.8117,0.7571,0.5814,0.6263,0.9157
5,xgboost_baseline,0.8077,0.7505,0.5687,0.6135,0.9109


In [12]:
# 16. ANALYZE CLASS-LEVEL PERFORMANCE

# Let's peek into the classification report of the BEST performing model based on F1-Score
best_model_name = df_results.iloc[0]['Model']
best_rep = reports_dict[best_model_name]

print(f"--- CLASS-LEVEL PERFORMANCE FOR: {best_model_name.upper()} ---")

# Convert dict report to DataFrame for nice printing, dropping 'accuracy' and average rows temporarily
df_class = pd.DataFrame(best_rep).T
df_class = df_class.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')

display(df_class.round(4))

print("\nObservation Note: Check the 'recall' and 'f1-score' of minority classes. A good model maintains balance across all categories.")


--- CLASS-LEVEL PERFORMANCE FOR: LINEAR_SVM_BASELINE ---


,precision,recall,f1-score,support
harassment,0.4512,0.3951,0.4213,772.0
hate_speech,0.7634,0.7948,0.7788,1072.0
insult,0.6038,0.5992,0.6015,262.0
normal,0.8681,0.8789,0.8735,4337.0



Observation Note: Check the 'recall' and 'f1-score' of minority classes. A good model maintains balance across all categories.


In [13]:
# 17. SELECT FINAL MODEL

print("--- FINAL MODEL SELECTION CRITERIA ---")
print("1. The primary metric is Macro F1-Score to prioritize fairness across imbalanced classes.")
print("2. The model with the highest Macro F1-Score on the Test Set is selected.\n")

final_selected_model = df_results.iloc[0]['Model']
final_f1 = df_results.iloc[0]['F1-Score']

print(f">>> THE SELECTED FINAL MODEL IS: **{final_selected_model}** <<<")
print(f"Achieving a Macro F1-Score of {final_f1:.4f}")


--- FINAL MODEL SELECTION CRITERIA ---
1. The primary metric is Macro F1-Score to prioritize fairness across imbalanced classes.
2. The model with the highest Macro F1-Score on the Test Set is selected.

>>> THE SELECTED FINAL MODEL IS: **linear_svm_baseline** <<<
Achieving a Macro F1-Score of 0.6688


In [14]:
# 18. SAVE EVALUATION RESULTS

csv_path = REPORTS_DIR / "model_evaluation_results.csv"
df_results.to_csv(csv_path, index=False)
print(f"Saved master evaluation results to: {csv_path}")

json_rep_path = REPORTS_DIR / "classification_reports.json"
with open(json_rep_path, 'w') as f:
    json.dump(reports_dict, f, indent=4)
print(f"Saved detailed classification reports to: {json_rep_path}")


Saved master evaluation results to: /home/zapp/Kampus/PM-NEW/reports/model_evaluation_results.csv
Saved detailed classification reports to: /home/zapp/Kampus/PM-NEW/reports/classification_reports.json


In [15]:
# 19. SAVE VISUALIZATIONS

# Visualizations (Confusion Matrices) were already saved in Step 14 to avoid memory overhead.
print(f"Verified {len(cm_paths)} confusion matrices saved in {REPORTS_DIR}")


Verified 6 confusion matrices saved in /home/zapp/Kampus/PM-NEW/reports


In [16]:
# 20. SAVE MODEL SELECTION METADATA

selection_metadata = {
    "selected_model": final_selected_model,
    "target": "cyberbullying_type",
    "selection_criteria": "Highest Macro F1-Score on unseen test data",
    "final_metrics": {
        "accuracy": float(df_results.iloc[0]['Accuracy']),
        "precision": float(df_results.iloc[0]['Precision']),
        "recall": float(df_results.iloc[0]['Recall']),
        "f1_score": float(df_results.iloc[0]['F1-Score']),
        "roc_auc": float(df_results.iloc[0]['ROC-AUC']) if pd.notnull(df_results.iloc[0]['ROC-AUC']) else None
    },
    "reason": "Demonstrated the best balance in classifying both majority and minority cyberbullying types without overfitting."
}

meta_path = REPORTS_DIR / "model_selection.json"
with open(meta_path, 'w') as f:
    json.dump(selection_metadata, f, indent=4)

print(f"Model selection metadata saved to {meta_path}")


Model selection metadata saved to /home/zapp/Kampus/PM-NEW/reports/model_selection.json


# 21. EVALUATION SUMMARY

### Models Evaluated
- Logistic Regression (Baseline & Tuned)
- Linear SVC (Baseline & Tuned)
- XGBoost (Baseline & Tuned)

### Metrics Used
- **Primary**: Macro F1-Score
- **Secondary**: Macro Precision, Macro Recall, Accuracy, ROC-AUC (OvR).

### Execution Integrity
This evaluation was executed entirely on `X_test_tfidf`, ensuring zero bias. All probabilities, confusion matrices, and metrics reflect real-world predictive capability.

### Saved Artifacts
- **DataFrames**: `model_evaluation_results.csv`
- **Detailed JSONs**: `classification_reports.json`, `model_selection.json`
- **Plots**: `confusion_matrix_*.png`

### Next Step
With the ultimate model identified, we can proceed to dissect its mistakes in `notebooks/10_error_analysis.ipynb` or understand its logic using XAI in `notebooks/11_explainability.ipynb`.


# 22. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the TF-IDF Test artifacts exist in: `data/processed/tfidf/` (specifically `X_test_tfidf.npz`).
3. Ensure model training and tuning have been completed in `notebooks/07_model_training.ipynb` and `notebooks/08_hyperparameter_tuning.ipynb`.
4. Check the `models/` folder to ensure all 6 `.pkl` files exist.
5. Open: `notebooks/09_model_evaluation.ipynb`.
6. Run the notebook from the first cell to the last cell.
7. Review the **Master Comparison DataFrame in Step 15**. This is the core result of your research!
8. Open the `reports/` folder in your file explorer to view the generated Confusion Matrix images (`.png`).
9. Proceed to: `notebooks/10_error_analysis.ipynb`.
